# 镜像层拆分：每个文件 → 独立镜像层

将原始 Docker 镜像层 `.tar.gz` 中的**每一个文件**提取出来，
重新打包为**独立的 `.tar.gz` 镜像层**。

这对应 New 仓库 "File-as-Layer" 的核心理念：  
- 原始：1 个 layer 包含 N 个文件  
- 转换后：N 个 layer，每个只包含 1 个文件  

**用途**：为离线去重测试准备数据，使每个文件成为独立的去重单元。

In [ ]:
import os
import tarfile
import hashlib
import gzip
import io
import time
from collections import defaultdict

print("✅ 库导入完成")

## 1. 配置

In [ ]:
# ============================================================
# ⚙️ 配置区 — 根据环境修改
# ============================================================

# 原始镜像层目录 (存放 Docker layer .tar.gz 文件)
input_dir = '/root/experiment/data/nginx_layers/'

# 输出目录 (每个文件打包为独立 .tar.gz)
output_dir = '/root/experiment/data/nginx_file_as_layer/'

# 是否跳过空文件 (符号链接/硬链接仍保留)
skip_empty = False

# 是否按文件内容 SHA256 去重 (相同内容只生成一个 layer)
dedup_output = True

print(f"📁 输入目录: {input_dir}  (存在: {os.path.isdir(input_dir)})")
print(f"📁 输出目录: {output_dir}")
print(f"🔧 跳过空文件: {skip_empty}")
print(f"🔧 输出去重:   {dedup_output}")

## 2. 查找所有原始镜像层

In [ ]:
def find_layers(layer_dir):
    """递归查找目录下所有 layer 文件 (tar.gz / 纯 tar / 纯 gz)"""
    layers = []
    for root, dirs, files in os.walk(layer_dir):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, 'rb') as f:
                    header = f.read(512)
                if len(header) < 2:
                    continue

                # 1) gzip magic → tar.gz 或 纯 .gz
                if header[:2] == b'\x1f\x8b':
                    layers.append((fpath, 'gzip'))
                # 2) tar magic "ustar" at offset 257
                elif len(header) >= 263 and header[257:262] == b'ustar':
                    layers.append((fpath, 'tar'))
                # 3) 尝试按名字匹配
                elif fname.endswith(('.tar.gz', '.tgz', '.tar', '.gz')):
                    layers.append((fpath, 'unknown'))
            except:
                pass
    return layers

layers = find_layers(input_dir)
print(f"📦 发现 {len(layers)} 个 layer 文件")
type_counts = defaultdict(int)
for _, t in layers:
    type_counts[t] += 1
print(f"   格式分布: {dict(type_counts)}")

for i, (lp, lt) in enumerate(layers[:5]):
    sz = os.path.getsize(lp)
    print(f"  [{i+1}] {os.path.basename(lp)[:40]}  ({sz/1024/1024:.2f} MiB, {lt})")
if len(layers) > 5:
    print(f"  ... 共 {len(layers)} 个")

## 3. 拆分：每个文件 → 独立 .tar.gz 镜像层

对每个原始 layer 执行：
1. 打开 `.tar.gz` 遍历所有成员
2. 对每个文件（普通文件 / 符号链接 / 硬链接），单独打包为一个 `.tar.gz`
3. 文件名格式：`{content_sha256_prefix}_{original_path_hash}.tar.gz`
4. 如果开启去重，相同内容的文件只生成一次

In [ ]:
# 清空旧输出（避免残留上次运行的短文件名文件）
import shutil
if os.path.isdir(output_dir):
    old_count = len(os.listdir(output_dir))
    shutil.rmtree(output_dir)
    print(f"🗑️ 已清空旧输出目录 ({old_count} 个旧文件)")
os.makedirs(output_dir, exist_ok=True)

stats = {
    'total_layers': len(layers),
    'total_files': 0,
    'total_dirs': 0,
    'total_symlinks': 0,
    'total_hardlinks': 0,
    'output_layers': 0,
    'dedup_skipped': 0,
    'empty_skipped': 0,
    'pure_gz_files': 0,       # 纯 .gz（非 tar）的文件数
    'total_input_bytes': 0,
    'total_output_bytes': 0,
}

seen_digests = set()  # 用于输出去重

# 记录每个输出 layer 的信息
output_records = []


def _process_file_content(content, member_name, member_mode, member_uid,
                          member_gid, member_mtime, layer_name, file_type='file'):
    """处理单个文件内容：去重检查 + 打包为独立 .tar.gz"""
    if skip_empty and len(content) == 0:
        stats['empty_skipped'] += 1
        return

    content_hash = hashlib.sha256(content).hexdigest()

    if dedup_output and content_hash in seen_digests:
        stats['dedup_skipped'] += 1
        output_records.append({
            'source_layer': layer_name,
            'tar_path': member_name,
            'type': file_type,
            'size': len(content),
            'content_hash': content_hash,
            'output_file': f'{content_hash}.tar.gz',
            'is_dedup': True,
        })
        return

    seen_digests.add(content_hash)

    out_name = f'{content_hash}.tar.gz'
    out_path = os.path.join(output_dir, out_name)

    with tarfile.open(out_path, 'w:gz') as out_tar:
        info = tarfile.TarInfo(name=member_name)
        info.size = len(content)
        info.mode = member_mode if member_mode else 0o644
        info.uid = member_uid if member_uid else 0
        info.gid = member_gid if member_gid else 0
        info.mtime = member_mtime if member_mtime else 0
        info.type = tarfile.REGTYPE
        out_tar.addfile(info, io.BytesIO(content))

    out_size = os.path.getsize(out_path)
    stats['output_layers'] += 1
    stats['total_output_bytes'] += out_size

    output_records.append({
        'source_layer': layer_name,
        'tar_path': member_name,
        'type': file_type,
        'size': len(content),
        'content_hash': content_hash,
        'output_file': out_name,
        'is_dedup': False,
    })


def _process_tar_archive(tar, layer_name):
    """处理一个 tar 归档中的所有成员"""
    members = tar.getmembers()

    for member in members:
        # === 目录：只统计，不生成独立 layer ===
        if member.isdir():
            stats['total_dirs'] += 1
            continue

        # === 普通文件 ===
        if member.isfile():
            stats['total_files'] += 1
            f = tar.extractfile(member)
            if f is None:
                continue
            content = f.read()
            _process_file_content(
                content, member.name, member.mode, member.uid,
                member.gid, member.mtime, layer_name, 'file')

        # === 符号链接 ===
        elif member.issym():
            stats['total_symlinks'] += 1
            pseudo = f'symlink:{member.linkname}'.encode()
            content_hash = hashlib.sha256(pseudo).hexdigest()

            if dedup_output and content_hash in seen_digests:
                stats['dedup_skipped'] += 1
                output_records.append({
                    'source_layer': layer_name,
                    'tar_path': member.name,
                    'type': 'symlink',
                    'size': 0,
                    'content_hash': content_hash,
                    'output_file': f'{content_hash}.tar.gz',
                    'is_dedup': True,
                })
                continue

            seen_digests.add(content_hash)

            out_name = f'{content_hash}.tar.gz'
            out_path = os.path.join(output_dir, out_name)

            with tarfile.open(out_path, 'w:gz') as out_tar:
                info = tarfile.TarInfo(name=member.name)
                info.type = tarfile.SYMTYPE
                info.linkname = member.linkname
                info.mode = member.mode
                out_tar.addfile(info)

            out_size = os.path.getsize(out_path)
            stats['output_layers'] += 1
            stats['total_output_bytes'] += out_size

            output_records.append({
                'source_layer': layer_name,
                'tar_path': member.name,
                'type': 'symlink',
                'size': 0,
                'content_hash': content_hash,
                'output_file': out_name,
                'is_dedup': False,
            })

        # === 硬链接 ===
        elif member.islnk():
            stats['total_hardlinks'] += 1
            pseudo = f'hardlink:{member.linkname}'.encode()
            content_hash = hashlib.sha256(pseudo).hexdigest()

            if dedup_output and content_hash in seen_digests:
                stats['dedup_skipped'] += 1
                output_records.append({
                    'source_layer': layer_name,
                    'tar_path': member.name,
                    'type': 'hardlink',
                    'size': 0,
                    'content_hash': content_hash,
                    'output_file': f'{content_hash}.tar.gz',
                    'is_dedup': True,
                })
                continue

            seen_digests.add(content_hash)

            out_name = f'{content_hash}.tar.gz'
            out_path = os.path.join(output_dir, out_name)

            with tarfile.open(out_path, 'w:gz') as out_tar:
                info = tarfile.TarInfo(name=member.name)
                info.type = tarfile.LNKTYPE
                info.linkname = member.linkname
                info.mode = member.mode
                out_tar.addfile(info)

            out_size = os.path.getsize(out_path)
            stats['output_layers'] += 1
            stats['total_output_bytes'] += out_size

            output_records.append({
                'source_layer': layer_name,
                'tar_path': member.name,
                'type': 'hardlink',
                'size': 0,
                'content_hash': content_hash,
                'output_file': out_name,
                'is_dedup': False,
            })


def _process_pure_gz(layer_path, layer_name):
    """
    Fallback: 处理纯 .gz 文件（gzip 压缩单个文件，内部不是 tar）。
    将解压后的整个内容作为一个文件打包成独立 .tar.gz。
    """
    with gzip.open(layer_path, 'rb') as gz:
        content = gz.read()

    stats['pure_gz_files'] += 1
    stats['total_files'] += 1

    # 用原始文件名去掉 .gz 后缀作为内部路径名
    inner_name = layer_name
    for ext in ('.tar.gz', '.tgz', '.gz'):
        if inner_name.endswith(ext):
            inner_name = inner_name[:-len(ext)]
            break
    if not inner_name:
        inner_name = 'data'

    _process_file_content(
        content, inner_name, 0o644, 0, 0, int(os.path.getmtime(layer_path)),
        layer_name, 'pure_gz')


# === 主处理循环 ===
start_time = time.time()

for li, (layer_path, layer_type) in enumerate(layers):
    layer_name = os.path.basename(layer_path)
    stats['total_input_bytes'] += os.path.getsize(layer_path)

    processed = False

    # 尝试 1: 作为 tar.gz 打开
    if layer_type in ('gzip', 'unknown'):
        try:
            with tarfile.open(layer_path, 'r:gz') as tar:
                _process_tar_archive(tar, layer_name)
            processed = True
        except (tarfile.ReadError, tarfile.CompressionError):
            pass  # 不是 tar.gz，继续尝试

    # 尝试 2: 作为纯 tar 打开（无 gzip 压缩）
    if not processed and layer_type in ('tar', 'unknown'):
        try:
            with tarfile.open(layer_path, 'r:') as tar:
                _process_tar_archive(tar, layer_name)
            processed = True
        except (tarfile.ReadError, tarfile.CompressionError):
            pass

    # 尝试 3: 作为纯 .gz 打开（gzip 压缩的单个文件，非 tar）
    if not processed and layer_type in ('gzip', 'unknown'):
        try:
            _process_pure_gz(layer_path, layer_name)
            processed = True
        except Exception:
            pass

    if not processed:
        print(f"\n  ⚠️ 无法处理 {layer_name} (类型: {layer_type})")

    elapsed = time.time() - start_time
    print(f"  [{li+1}/{len(layers)}] {layer_name[:30]}... → "
          f"{stats['output_layers']} 个独立层  ({elapsed:.1f}s)", end='\r')

elapsed = time.time() - start_time
print(f"\n\n✅ 拆分完成! 耗时 {elapsed:.1f}s")
print(f"\n=== 统计 ===")
print(f"原始镜像层数:    {stats['total_layers']:>8,}")
print(f"总文件数:        {stats['total_files']:>8,}")
print(f"  其中纯 .gz:    {stats['pure_gz_files']:>8,}  (非 tar 归档)")
print(f"总目录数:        {stats['total_dirs']:>8,}  (不生成独立层)")
print(f"总符号链接:      {stats['total_symlinks']:>8,}")
print(f"总硬链接:        {stats['total_hardlinks']:>8,}")
print(f"输出独立层数:    {stats['output_layers']:>8,}")
if dedup_output:
    print(f"去重跳过:        {stats['dedup_skipped']:>8,}")
if skip_empty:
    print(f"空文件跳过:      {stats['empty_skipped']:>8,}")
print(f"输入总大小:      {stats['total_input_bytes']/1024**2:>10.2f} MiB")
print(f"输出总大小:      {stats['total_output_bytes']/1024**2:>10.2f} MiB")
print(f"\n📁 输出目录: {output_dir}")

## 4. 拆分结果分析

In [ ]:
import pandas as pd

df = pd.DataFrame(output_records)
print(f"总记录数: {len(df)}")
print(f"\n--- 按类型统计 ---")
print(df.groupby('type').agg(
    count=('type', 'size'),
    total_size=('size', 'sum'),
    unique=('is_dedup', lambda x: (~x).sum()),
    dedup=('is_dedup', 'sum'),
))

print(f"\n--- 按源 layer 统计 ---")
layer_stats = df.groupby('source_layer').agg(
    total_files=('tar_path', 'count'),
    unique_files=('is_dedup', lambda x: (~x).sum()),
    dedup_files=('is_dedup', 'sum'),
    total_bytes=('size', 'sum'),
)
display(layer_stats)

print(f"\n--- 文件大小分布 ---")
files_only = df[df['type'] == 'file']
if len(files_only) > 0:
    bins = [0, 1024, 10*1024, 100*1024, 512*1024, 1024*1024, 10*1024*1024, float('inf')]
    labels = ['<1K', '1K-10K', '10K-100K', '100K-512K', '512K-1M', '1M-10M', '>10M']
    files_only = files_only.copy()
    files_only['size_bin'] = pd.cut(files_only['size'], bins=bins, labels=labels)
    size_dist = files_only.groupby('size_bin', observed=True).agg(
        count=('size', 'count'),
        total_bytes=('size', 'sum'),
    )
    size_dist['total_MiB'] = (size_dist['total_bytes'] / 1024**2).round(2)
    display(size_dist[['count', 'total_MiB']])

## 5. 导出映射表

导出 CSV 记录每个输出 layer 与原始文件的对应关系。

In [ ]:
csv_path = os.path.join(output_dir, 'file_to_layer_mapping.csv')
df.to_csv(csv_path, index=False)
print(f"✅ 映射表已保存: {csv_path}")
print(f"   共 {len(df)} 条记录")
print(f"   唯一输出层: {stats['output_layers']} 个 .tar.gz 文件")

# 验证输出目录
out_files = [f for f in os.listdir(output_dir) if f.endswith('.tar.gz')]
print(f"\n📁 输出目录 {output_dir} 包含 {len(out_files)} 个 .tar.gz 文件")
print(f"   总大小: {sum(os.path.getsize(os.path.join(output_dir, f)) for f in out_files)/1024**2:.2f} MiB")

print(f"\n🎉 完成! 可将 {output_dir} 作为 file_dedup_ratio.ipynb 的 layer_dir 输入。")